# R1-01 — Multi-seed retraining (camera-ready, ICINCO 2026 paper #122)

Retrains the fusion model with several seeds on **MSILN site1/B1** and **IMUWiFine floor 4**,
plus **WiFi-Net on UJIIndoorLoc**, at the paper configuration (K=4, 40 epochs, batch 128,
modality-balanced loss off). Per-run `summary.json` + checkpoints are written to Google Drive,
so the notebook is **safe to re-run after a disconnect** (finished runs are skipped).

Flow: params -> Drive mount -> clone + imports -> data (MSILN from the Microsoft starter repo,
IMUWiFine from HuggingFace, UJI from UCI) -> training loops -> aggregate table to paste back.

Use a GPU runtime (Runtime > Change runtime type > T4 GPU). Rough total for 3 seeds: 2-3 h.


In [ ]:
# ==== Parameters ====
SEEDS = [42, 7, 123]        # add more seeds here later if we decide to
EPOCHS = 40                 # paper config (Sec 5.4)
K = 4                       # n_instants, paper config
BATCH = 128                 # paper config
MBL = False                 # modality_balanced_loss off = paper config
DATASETS = ["msiln_site1_b1", "imuwifine"]
RUN_UJI = True              # 3-seed WiFi-Net on UJIIndoorLoc (Table 3 row)
UJI_EPOCHS = 120            # eval_uji_wifi.py default (the 8.69 m protocol)
DRIVE_DIR = "navlori_camera_ready"


In [ ]:
# ==== Google Drive ====
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
OUT_ROOT = Path("/content/drive/MyDrive") / DRIVE_DIR
(OUT_ROOT / "runs").mkdir(parents=True, exist_ok=True)
print("results root:", OUT_ROOT)


In [ ]:
# ==== Clone repo + self-healing imports ====
import os, sys, subprocess, json, time, random, io, re
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (enable a GPU runtime!)")

REPO = Path("/content/navlori-fusion")
if not REPO.exists():
    subprocess.check_call(["git", "clone", "--depth", "1",
                           "https://github.com/moebachar/navlori-fusion.git", str(REPO)])
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# src.pipeline.encoders imports baselines eagerly, which needs these two submodules
# (ronin: model_resnet1d at import time; dpvo: BasicEncoder4 resolved at module top)
def _run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if out.strip(): print(out.strip()[-2000:])
    return r.returncode

SUBMODULES = {
    "external_methods/ronin": ("https://github.com/Sachini/ronin",
                               "805b7f0f28bb164ce89ada9ac05a9470dbe3d715",
                               "source/model_resnet1d.py"),
    "external_methods/dpvo": ("https://github.com/princeton-vl/DPVO",
                              "859bbbfdac6c6185f345003b3c473901fcd13ace",
                              "dpvo/extractor.py"),
}
print("-- git submodule update --init --")
_run(["git", "submodule", "update", "--init"] + list(SUBMODULES), cwd=str(REPO))
import shutil
for path, (url, sha, marker) in SUBMODULES.items():
    d = REPO / path
    if not (d / marker).exists():
        print(f"-- submodule route did not materialize {path}; direct clone fallback --")
        if d.exists():
            shutil.rmtree(d, ignore_errors=True)
        _run(["git", "clone", url, str(d)])
        _run(["git", "checkout", sha], cwd=str(d))
    assert (d / marker).exists(), f"{path}/{marker} STILL missing - save a copy to GitHub and tell Claude"
    print(f"OK: {path} ({marker} present)")

PIPNAME = {"omegaconf": "omegaconf", "yaml": "pyyaml", "sklearn": "scikit-learn",
           "cv2": "opencv-python-headless", "mlflow": "mlflow", "torchdiffeq": "torchdiffeq",
           "seaborn": "seaborn", "influxdb_client": "influxdb-client", "plotly": "plotly",
           "quaternion": "numpy-quaternion", "PIL": "pillow", "skimage": "scikit-image"}
# a stray PyPI package literally named 'quaternion' shadows nothing but confuses installs
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "quaternion"],
               capture_output=True)
_tried = set()
for attempt in range(10):
    try:
        from src.pipeline.fusion.builder import (build_datamodule, build_encoders,
                                                 build_model, build_trainer, load_config)
        print("imports OK"); break
    except ModuleNotFoundError as e:
        pkg = e.name.split(".")[0]
        if pkg in _tried:
            raise RuntimeError(
                f"module '{e.name}' still missing after installing '{PIPNAME.get(pkg, pkg)}' - "
                "save a copy to GitHub and tell Claude.") from e
        _tried.add(pkg)
        pip = PIPNAME.get(pkg, pkg)
        print("missing module:", e.name, "-> pip install", pip)
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip])
        if r.returncode != 0:
            raise RuntimeError(
                f"'{e.name}' is not pip-installable - likely a repo-local module. "
                "Save a copy of this notebook to GitHub and tell Claude.") from e
else:
    raise RuntimeError("imports still failing after installs - send Claude the error above")

import numpy as np

def set_global_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


## Data — MSILN site1/B1

The converter reads the cloned Microsoft starter repository
(`indoor-location-competition-20`), which ships the site1/B1 trace files.
If the clone turns out not to contain `data/site1/B1` (message below will say so),
stop and tell Claude — the fallback goes through the Kaggle competition download instead.


In [ ]:
# ==== Dataset cache helpers (Drive) + MSILN ====
import shutil
DATA_CACHE = OUT_ROOT / "data_cache"

def run_logged(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if out.strip(): print(out.strip()[-3000:])
    return r.returncode

def drive_restore(name, marker="metadata.json"):
    dst = REPO / "data" / name
    if (dst / marker).exists(): return "already in session"
    src = DATA_CACHE / name
    if (src / marker).exists():
        shutil.copytree(src, dst, dirs_exist_ok=True); return "restored from Drive cache"
    return None

def drive_cache(name, marker="metadata.json"):
    src = REPO / "data" / name; dst = DATA_CACHE / name
    if (src / marker).exists() and not (dst / marker).exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"cached data/{name} to Drive")

st = drive_restore("msiln_site1_b1")
if st:
    print("MSILN:", st)
else:
    STARTER = Path("/content/indoor-location-competition-20")
    if not STARTER.exists():
        subprocess.check_call(["git", "clone", "--depth", "1",
            "https://github.com/location-competition/indoor-location-competition-20.git",
            str(STARTER)])
    site_dir = STARTER / "data" / "site1" / "B1"
    if not site_dir.exists():
        print("!!! data/site1/B1 not found in the starter repo. Top of its data/ tree:")
        for p in sorted((STARTER / "data").glob("*"))[:20]:
            print("   ", p.name)
        raise RuntimeError("site1/B1 missing - tell Claude, we will switch to the Kaggle route")
    if run_logged([sys.executable, "scripts/convert_msiln.py", "--msiln-root", str(STARTER),
                   "--site", "site1", "--floor", "B1", "--out-root", "data"]) != 0:
        raise RuntimeError("convert_msiln.py failed - output above")
    drive_cache("msiln_site1_b1")
n_paths = len(list((REPO / "data" / "msiln_site1_b1").glob("path_*")))
print(f"MSILN ready: {n_paths} paths (expected 133)")


In [ ]:
# ==== IMUWiFine: download from HuggingFace + convert (floor 4) ====
st = drive_restore("imuwifine_floor4")
if st:
    print("IMUWiFine:", st)
else:
    from huggingface_hub import snapshot_download
    raw = Path(snapshot_download(repo_id="issai/IMUWiFine", repo_type="dataset",
                                 local_dir="/content/IMUWiFine_raw"))
    # converter code expects: <raw_root>/IMU_DATA/raw_IUMIWiFi/<N>th_floor/{train,val}
    #                    and: <raw_root>/IMU_DATA/test/test_<N>_*.txt
    hits = [p for p in raw.rglob("raw_IUMIWiFi") if p.is_dir()]
    raw_root = None
    if hits:
        p = hits[0]
        if p.parent.name == "IMU_DATA":
            raw_root = p.parent.parent
        else:
            stage = Path("/content/iw_root")
            (stage / "IMU_DATA").mkdir(parents=True, exist_ok=True)
            link = stage / "IMU_DATA" / "raw_IUMIWiFi"
            if not link.exists(): link.symlink_to(p)
            tests = [t for t in raw.rglob("test") if t.is_dir() and list(t.glob("test_*.txt"))]
            if tests:
                tl = stage / "IMU_DATA" / "test"
                if not tl.exists(): tl.symlink_to(tests[0])
            raw_root = stage
    ok = raw_root and (Path(raw_root) / "IMU_DATA" / "raw_IUMIWiFi" / "4th_floor").exists() \
         and (Path(raw_root) / "IMU_DATA" / "test").exists()
    if not ok:
        print("!!! layout not resolvable. Downloaded tree (2 levels):")
        for p in sorted(raw.glob("*")):
            print("   ", p.name)
            if p.is_dir():
                for q in sorted(p.glob("*"))[:12]:
                    print("       ", q.name)
        raise RuntimeError("unexpected IMUWiFine layout - paste the tree above to Claude")
    if run_logged([sys.executable, "scripts/convert_imuwifine.py", "--floor", "4",
                   "--raw-root", str(raw_root), "--out-root", "data"]) != 0:
        raise RuntimeError("convert_imuwifine.py failed - output above")
    drive_cache("imuwifine_floor4")
n_paths = len(list((REPO / "data" / "imuwifine_floor4").glob("path_*")))
print(f"IMUWiFine floor 4 ready: {n_paths} paths (expected 80: 40 train / 20 val / 20 test)")


## Training — fusion model, one run per (dataset, seed)

Paper config enforced explicitly: K=4, 40 epochs, batch 128, modality-balanced loss off
(the repo YAML defaults differ - K=8/90 epochs/MBL on - so the overrides below matter).
Each finished run leaves `summary.json` + `model_last.pt` in Drive and is skipped on re-runs.


In [ ]:
# ==== Fusion training loop ====
RESULTS = []
for ds in DATASETS:
    for seed in SEEDS:
        run_name = f"{ds}_seed{seed}"
        run_root = OUT_ROOT / "runs" / run_name
        summ_p = run_root / "summary.json"
        if summ_p.exists():
            s = json.loads(summ_p.read_text()); RESULTS.append(s)
            print(f"skip {run_name}: val {s['val_mae_m']:.2f}  test {s['test_mae_m']:.2f}")
            continue
        run_root.mkdir(parents=True, exist_ok=True)
        print(f"\n===== {run_name} =====", flush=True)
        set_global_seed(seed)
        cfg = load_config(ds)
        cfg.temporal.n_instants = K
        cfg.train.modality_balanced_loss = MBL
        cfg.data.batch_size = BATCH
        dm = build_datamodule(cfg)
        encs, vision = build_encoders(cfg, dm)
        model = build_model(cfg, encs)
        trainer = build_trainer(cfg, model, dm, run_dir=str(run_root))
        t0 = time.time()
        hist = trainer.fit(epochs=EPOCHS)
        elapsed = time.time() - t0
        preds_v, tgts_v = trainer.predict("val")
        val_mae = float((preds_v - tgts_v).norm(dim=1).mean())
        test_mae = float("nan")
        if "test" in trainer.splits:
            preds_t, tgts_t = trainer.predict("test")
            test_mae = float((preds_t - tgts_t).norm(dim=1).mean())
        torch.save(trainer.model.state_dict(), run_root / "model_last.pt")
        summary = {"dataset": ds, "seed": seed, "epochs": EPOCHS, "K": K,
                   "batch": BATCH, "mbl": MBL,
                   "val_mae_m": val_mae, "test_mae_m": test_mae,
                   "best_val_mae_m": float(hist.best_val_mae),
                   "best_epoch": int(hist.best_epoch), "elapsed_s": elapsed}
        summ_p.write_text(json.dumps(summary, indent=2))
        RESULTS.append(summary)
        print("SUMMARY:", json.dumps(summary), flush=True)
        del trainer, model, dm, encs
        torch.cuda.empty_cache()
print(f"\nfusion runs complete: {len(RESULTS)}")


In [ ]:
# ==== UJI: data + 3-seed WiFi-Net (Table 3 row) ====
UJI_RESULTS = []
if RUN_UJI:
    UJI_DIR = REPO / "data" / "uji_indoorloc"
    if not (UJI_DIR / "trainingData.csv").exists():
        UJI_DIR.mkdir(parents=True, exist_ok=True)
        import urllib.request, zipfile
        zpath = Path("/content/ujiindoorloc.zip")
        if not zpath.exists():
            urllib.request.urlretrieve(
                "https://archive.ics.uci.edu/static/public/310/ujiindoorloc.zip", zpath)
        with zipfile.ZipFile(zpath) as z:
            z.extractall("/content/uji_tmp")
        found = {}
        for p in Path("/content/uji_tmp").rglob("*.csv"):
            n = p.name.lower()
            if n == "trainingdata.csv": found["trainingData.csv"] = p
            if n == "validationdata.csv": found["validationData.csv"] = p
        assert len(found) == 2, f"UJI csvs not found in zip: {found}"
        import shutil
        for name, p in found.items():
            shutil.copy(p, UJI_DIR / name)
    print("UJI data ready")

    import importlib.util, contextlib
    spec = importlib.util.spec_from_file_location("uji_wifi", REPO / "scripts" / "eval_uji_wifi.py")
    uji_mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(uji_mod)

    class Tee(io.TextIOBase):
        def __init__(self, *streams): self.streams = streams
        def write(self, s):
            for st in self.streams: st.write(s)
            return len(s)
        def flush(self):
            for st in self.streams: st.flush()

    for seed in SEEDS:
        out_p = OUT_ROOT / "runs" / f"uji_wifinet_seed{seed}.json"
        if out_p.exists():
            r = json.loads(out_p.read_text()); UJI_RESULTS.append(r)
            print(f"skip uji seed {seed}: best {r['best_val_mae_m']:.2f}")
            continue
        print(f"\n===== UJI WiFi-Net seed {seed} =====", flush=True)
        set_global_seed(seed)
        buf = io.StringIO()
        sys.argv = ["eval_uji_wifi.py", "--epochs", str(UJI_EPOCHS)]
        with contextlib.redirect_stdout(Tee(sys.stdout, buf)):
            uji_mod.main()
        bests = re.findall(r"best ([0-9]+\.[0-9]+)", buf.getvalue())
        best = float(bests[-1]) if bests else float("nan")
        r = {"dataset": "uji_indoorloc", "seed": seed, "epochs": UJI_EPOCHS,
             "best_val_mae_m": best}
        out_p.write_text(json.dumps(r, indent=2))
        UJI_RESULTS.append(r)
        torch.cuda.empty_cache()


In [ ]:
# ==== Aggregate: mean +/- std (sample std) -> paste this back ====
import statistics
def agg(vals):
    vals = [v for v in vals if v == v]  # drop NaN
    if len(vals) < 2:
        return (vals[0] if vals else float("nan")), 0.0
    return statistics.mean(vals), statistics.stdev(vals)

report = {"seeds": SEEDS, "config": {"epochs": EPOCHS, "K": K, "batch": BATCH, "mbl": MBL},
          "fusion": {}, "uji_wifinet": {}}
for ds in DATASETS:
    rs = [r for r in RESULTS if r["dataset"] == ds]
    vm, vs = agg([r["val_mae_m"] for r in rs])
    tm, ts = agg([r["test_mae_m"] for r in rs])
    report["fusion"][ds] = {"n": len(rs),
        "val_mae": f"{vm:.2f} +/- {vs:.2f}", "test_mae": f"{tm:.2f} +/- {ts:.2f}",
        "per_seed_test": {r["seed"]: round(r["test_mae_m"], 2) for r in rs}}
if UJI_RESULTS:
    m, s = agg([r["best_val_mae_m"] for r in UJI_RESULTS])
    report["uji_wifinet"] = {"n": len(UJI_RESULTS), "val_mae": f"{m:.2f} +/- {s:.2f}",
        "per_seed": {r["seed"]: round(r["best_val_mae_m"], 2) for r in UJI_RESULTS}}

print("=" * 60)
print("PASTE EVERYTHING BELOW BACK TO CLAUDE")
print("=" * 60)
print(json.dumps(report, indent=2))


## Done

Copy the JSON block above back into the Claude session. The per-run folders in
`Drive/{DRIVE_DIR}/runs/` (checkpoints + summaries) are reused later for the
all-paths figures (R1-02, R2-05) — do not delete them.
